# stable-marl

> The public API: environments, the `World` that runs policies in them, and the datasets it collects.

As in stable-worldmodel, submodules and top-level attributes load lazily (PEP 562), so
`import stable_marl` stays fast: torch, h5py and the planning / world model code are imported
on first access (e.g. `stable_marl.planning`, `stable_marl.World`). Only `stable_marl.envs`
is imported eagerly: it registers the environment ids `make` needs.

In [ ]:
#| default_exp __init__

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import importlib
from typing import TYPE_CHECKING

from stable_marl import envs   # eager: registers the environments
from stable_marl.envs import MultiAgentEnv, make, list_envs, register

_LAZY_SUBMODULES = {'buffer', 'configs', 'data', 'planning', 'plot', 'policy', 'protocols', 'spaces', 'types',
                    'utils', 'wm', 'world', 'wrappers'}

_LAZY_ATTRS = {
    'World': ('stable_marl.world', 'World'),
    'EnvPool': ('stable_marl.world', 'EnvPool'),
    'BasePolicy': ('stable_marl.policy', 'BasePolicy'),
    'Policy': ('stable_marl.policy', 'Policy'),
    'RandomPolicy': ('stable_marl.policy', 'RandomPolicy'),
    'ExpertPolicy': ('stable_marl.policy', 'ExpertPolicy'),
    'FeedForwardPolicy': ('stable_marl.policy', 'FeedForwardPolicy'),
    'PlanConfig': ('stable_marl.policy', 'PlanConfig'),
    'WorldModelPolicy': ('stable_marl.policy', 'WorldModelPolicy'),
    'HDF5Dataset': ('stable_marl.data', 'HDF5Dataset'),
}

if TYPE_CHECKING:   # for editors and type checkers
    from stable_marl import buffer, configs, data, planning, plot, policy, protocols, spaces, types, utils, wm, world, wrappers
    from stable_marl.data import HDF5Dataset
    from stable_marl.policy import (BasePolicy, Policy, RandomPolicy, ExpertPolicy, FeedForwardPolicy,
                                    PlanConfig, WorldModelPolicy)
    from stable_marl.world import World, EnvPool


def __getattr__(name: str):
    if name in _LAZY_SUBMODULES:
        mod = importlib.import_module(f'.{name}', __name__)
        globals()[name] = mod
        return mod
    if name in _LAZY_ATTRS:
        modpath, attrname = _LAZY_ATTRS[name]
        attr = getattr(importlib.import_module(modpath), attrname)
        globals()[name] = attr
        return attr
    raise AttributeError(f'module {__name__!r} has no attribute {name!r}')


def __dir__() -> list[str]:
    return sorted(set(__all__) | set(globals().keys()))

_all_ = ['MultiAgentEnv', 'make', 'list_envs', 'register', 'World', 'EnvPool', 'BasePolicy', 'Policy',
         'RandomPolicy', 'ExpertPolicy', 'FeedForwardPolicy', 'PlanConfig', 'WorldModelPolicy', 'HDF5Dataset',
         'buffer', 'configs', 'data', 'envs', 'planning', 'plot', 'policy', 'protocols', 'spaces', 'types', 'utils', 'wm',
         'world', 'wrappers']

In [ ]:
#| hide
# fresh interpreter: only the envs load with the package, the rest on first access
import subprocess, sys
code = '''
import sys, stable_marl as sm
heavy = ('torch', 'h5py', 'stable_marl.world', 'stable_marl.planning', 'stable_marl.wm', 'stable_marl.data')
assert not [m for m in heavy if m in sys.modules], [m for m in heavy if m in sys.modules]
assert sm.make is sm.envs.make and 'World' in dir(sm)
assert sm.World.__module__ == 'stable_marl.world.world' and 'stable_marl.world' in sys.modules
assert sm.planning.CEMSolver and sm.wm.LeWM and 'torch' in sys.modules
from stable_marl import *
assert RandomPolicy is sm.policy.RandomPolicy and wrappers is sm.wrappers
try:
    sm.nope
    raise SystemExit('missing attribute must raise')
except AttributeError:
    pass
'''
subprocess.run([sys.executable, '-W', 'ignore', '-c', code], check=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()